# Create a Vector Store
---

In [ ]:
# Install Faiss (CPU version)
!pip install faiss-cpu --quiet

# Install the Mistral client
!pip install mistralai --quiet

In [ ]:
import os
# Read the API key from an environment variable (never hard-code a secret)
os.environ.setdefault("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY")

In [ ]:
from mistralai import Mistral
import os
import numpy as np

# Initialize the client with your API key
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

# Function to generate the embedding of a text
def embed_text(text):
    # Using the 'create' method of the 'embeddings' object
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[text]
    )
    return response.data[0].embedding

# Example documents to index
documents = [
    "City Hall is open Monday to Friday from 9 AM to 5 PM.",
    "The city regulations prohibit parking in front of schools.",
    "The recycling center hours vary by season.",
    "Passport applications are by appointment only.",
    "Social assistance is available subject to income conditions."
]

# Generate the embeddings for each document
embeddings = np.array([embed_text(doc) for doc in documents])

In [ ]:
import faiss

# Dimension of the vectors (embedding size)
dimension = embeddings.shape[1]

# Create an L2 Faiss index (Euclidean distance)
index = faiss.IndexFlatL2(dimension)

# Add the vectors to the index
index.add(embeddings)

# Optional save
faiss.write_index(index, "faiss_index.idx")

In [ ]:
# User query
query = "What are the City Hall opening hours?"

# Generate the embedding of the query
query_embedding = embed_text(query)

# Search for the 3 closest documents
k = 3
distances, indices = index.search(np.array([query_embedding]), k)

# Display the results
print("Most relevant results:\n")
for i, idx in enumerate(indices[0]):
    print(f"📝 Document {idx} (distance: {distances[0][i]:.4f}):\n{documents[idx]}\n")